In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, col
import os
import shutil
import time
import pandas as pd
from datetime import datetime

spark = SparkSession.builder \
    .appName("Tugas10") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/05 08:05:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


A.

In [3]:
skema = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True)
])

folder = "tugas10_stream"

shutil.rmtree(folder, ignore_errors=True)
os.makedirs(folder, exist_ok=True)

In [4]:
df_stream = spark.readStream \
    .format("csv") \
    .schema(skema) \
    .option("header", True) \
    .load(folder)

print(df_stream.isStreaming)

True


B.

In [5]:
df = df_stream.withColumn(
    "total_harga",
    col("qty") * col("harga_satuan")
)

hasil = df.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

query1 = hasil.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

C.

In [6]:
df2 = spark.readStream \
    .format("csv") \
    .schema(skema) \
    .option("header", True) \
    .load(folder)

df2 = df2.withColumn(
    "total_harga",
    col("qty") * col("harga_satuan")
)

hasil2 = df2.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_pendapatan")
)

query2 = hasil2.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

D.

In [7]:
pd.DataFrame({
    "pesanan_id": ["P001", "P002", "P003"],
    "nama_cabang": ["Magelang", "Yogyakarta", "Magelang"],
    "nama_menu": ["Kopi Susu", "Americano", "Cappuccino"],
    "qty": [2, 1, 3],
    "harga_satuan": [15000, 18000, 20000],
    "waktu_pesan": [datetime.now().strftime("%Y-%m-%d %H:%M:%S")] * 3
}).to_csv(f"{folder}/batch1.csv", index=False)

time.sleep(6)

spark.sql("SELECT * FROM agregasi_cabang").show()

+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
+-----------+----------------+--------------+



In [8]:
pd.DataFrame({
    "pesanan_id": ["P004", "P005", "P006"],
    "nama_cabang": ["Magelang", "Semarang", "Yogyakarta"],
    "nama_menu": ["Latte", "Kopi Susu", "Americano"],
    "qty": [1, 2, 2],
    "harga_satuan": [22000, 15000, 18000],
    "waktu_pesan": [datetime.now().strftime("%Y-%m-%d %H:%M:%S")] * 3
}).to_csv(f"{folder}/batch2.csv", index=False)

time.sleep(6)

spark.sql("SELECT * FROM agregasi_cabang").show()

[Stage 1:=========>         (2 + 2) / 4][Stage 3:==============>    (3 + 1) / 4]

+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
+-----------+----------------+--------------+



In [9]:
pd.DataFrame({
    "pesanan_id": ["P007", "P008", "P009"],
    "nama_cabang": ["Semarang", "Magelang", "Yogyakarta"],
    "nama_menu": ["Cappuccino", "Americano", "Latte"],
    "qty": [1, 2, 1],
    "harga_satuan": [20000, 18000, 22000],
    "waktu_pesan": [datetime.now().strftime("%Y-%m-%d %H:%M:%S")] * 3
}).to_csv(f"{folder}/batch3.csv", index=False)

time.sleep(6)

spark.sql("SELECT * FROM agregasi_cabang").show()

+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Semarang|           50000|             2|
| Yogyakarta|           76000|             3|
|   Magelang|          148000|             4|
+-----------+----------------+--------------+



E.

In [10]:
shutil.rmtree("tugas10_output", ignore_errors=True)
shutil.rmtree("tugas10_checkpoint", ignore_errors=True)

query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

In [11]:
pd.DataFrame({
    "pesanan_id": ["P010", "P011"],
    "nama_cabang": ["Magelang", "Semarang"],
    "nama_menu": ["Kopi Susu", "Latte"],
    "qty": [2, 1],
    "harga_satuan": [15000, 22000],
    "waktu_pesan": [datetime.now().strftime("%Y-%m-%d %H:%M:%S")] * 2
}).to_csv(f"{folder}/batch4.csv", index=False)

time.sleep(8)

In [12]:
!ls tugas10_output

df3 = spark.read.parquet("tugas10_output")
df3.show()

part-00000-1f386b98-de7f-4ca8-a933-3ded3fa6fa99-c000.snappy.parquet
part-00001-6606a3a3-a5ce-4b71-9497-afad9b796dd2-c000.snappy.parquet
part-00002-5b51631e-7d56-4c4d-aa5f-bd5a66cc5b6b-c000.snappy.parquet
part-00003-14e38b84-5f16-4d24-822c-45313faaa086-c000.snappy.parquet
_spark_metadata
+----------+-----------+----------+---+------------+-------------------+
|pesanan_id|nama_cabang| nama_menu|qty|harga_satuan|        waktu_pesan|
+----------+-----------+----------+---+------------+-------------------+
|      P001|   Magelang| Kopi Susu|  2|       15000|2026-11-05 08:06:16|
|      P002| Yogyakarta| Americano|  1|       18000|2026-11-05 08:06:16|
|      P003|   Magelang|Cappuccino|  3|       20000|2026-11-05 08:06:16|
|      P007|   Semarang|Cappuccino|  1|       20000|2026-11-05 08:06:28|
|      P008|   Magelang| Americano|  2|       18000|2026-11-05 08:06:28|
|      P009| Yogyakarta|     Latte|  1|       22000|2026-11-05 08:06:28|
|      P004|   Magelang|     Latte|  1|       22000|202

Eksplorasi

In [13]:
hasil = df_stream.dropDuplicates(["pesanan_id"])

query = hasil.writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("data_unik") \
    .start()

time.sleep(6)
spark.sql("SELECT * FROM data_unik").show()

query.stop()

+----------+-----------+----------+---+------------+-------------------+
|pesanan_id|nama_cabang| nama_menu|qty|harga_satuan|        waktu_pesan|
+----------+-----------+----------+---+------------+-------------------+
|      P002| Yogyakarta| Americano|  1|       18000|2026-11-05 08:06:16|
|      P004|   Magelang|     Latte|  1|       22000|2026-11-05 08:06:22|
|      P006| Yogyakarta| Americano|  2|       18000|2026-11-05 08:06:22|
|      P010|   Magelang| Kopi Susu|  2|       15000|2026-11-05 08:06:36|
|      P011|   Semarang|     Latte|  1|       22000|2026-11-05 08:06:36|
|      P003|   Magelang|Cappuccino|  3|       20000|2026-11-05 08:06:16|
|      P007|   Semarang|Cappuccino|  1|       20000|2026-11-05 08:06:28|
|      P009| Yogyakarta|     Latte|  1|       22000|2026-11-05 08:06:28|
|      P005|   Semarang| Kopi Susu|  2|       15000|2026-11-05 08:06:22|
|      P001|   Magelang| Kopi Susu|  2|       15000|2026-11-05 08:06:16|
|      P008|   Magelang| Americano|  2|       18000

In [14]:
hasil = df2.groupBy(
    window(col("waktu_pesan"), "2 minutes", "1 minute")
).agg(
    spark_sum("total_harga").alias("total_pendapatan")
)

query = hasil.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("window_geser") \
    .start()

time.sleep(6)
spark.sql("SELECT * FROM window_geser").show(truncate=False)

query.stop()

+------------------------------------------+----------------+
|window                                    |total_pendapatan|
+------------------------------------------+----------------+
|{2026-11-05 08:05:00, 2026-11-05 08:07:00}|326000          |
|{2026-11-05 08:06:00, 2026-11-05 08:08:00}|326000          |
+------------------------------------------+----------------+



In [15]:
query = df_stream.writeStream \
    .format("console") \
    .outputMode("append") \
    .trigger(processingTime="10 seconds") \
    .start()

time.sleep(20)
query.stop()

-------------------------------------------
Batch: 0
-------------------------------------------
+----------+-----------+----------+---+------------+-------------------+
|pesanan_id|nama_cabang| nama_menu|qty|harga_satuan|        waktu_pesan|
+----------+-----------+----------+---+------------+-------------------+
|      P001|   Magelang| Kopi Susu|  2|       15000|2026-11-05 08:06:16|
|      P002| Yogyakarta| Americano|  1|       18000|2026-11-05 08:06:16|
|      P003|   Magelang|Cappuccino|  3|       20000|2026-11-05 08:06:16|
|      P007|   Semarang|Cappuccino|  1|       20000|2026-11-05 08:06:28|
|      P008|   Magelang| Americano|  2|       18000|2026-11-05 08:06:28|
|      P009| Yogyakarta|     Latte|  1|       22000|2026-11-05 08:06:28|
|      P004|   Magelang|     Latte|  1|       22000|2026-11-05 08:06:22|
|      P005|   Semarang| Kopi Susu|  2|       15000|2026-11-05 08:06:22|
|      P006| Yogyakarta| Americano|  2|       18000|2026-11-05 08:06:22|
|      P010|   Magelang| Ko

In [16]:
query = df_stream.select(
    "pesanan_id", "nama_cabang", "nama_menu", "qty"
).writeStream \
    .format("console") \
    .outputMode("append") \
    .option("truncate", False) \
    .start()

time.sleep(10)
query.stop()

-------------------------------------------
Batch: 0
-------------------------------------------
+----------+-----------+----------+---+
|pesanan_id|nama_cabang|nama_menu |qty|
+----------+-----------+----------+---+
|P001      |Magelang   |Kopi Susu |2  |
|P002      |Yogyakarta |Americano |1  |
|P003      |Magelang   |Cappuccino|3  |
|P007      |Semarang   |Cappuccino|1  |
|P008      |Magelang   |Americano |2  |
|P009      |Yogyakarta |Latte     |1  |
|P004      |Magelang   |Latte     |1  |
|P005      |Semarang   |Kopi Susu |2  |
|P006      |Yogyakarta |Americano |2  |
|P010      |Magelang   |Kopi Susu |2  |
|P011      |Semarang   |Latte     |1  |
+----------+-----------+----------+---+



In [17]:
shutil.rmtree("tugas10_json", ignore_errors=True)
shutil.rmtree("tugas10_json_checkpoint", ignore_errors=True)

query = df_stream.writeStream \
    .format("json") \
    .option("path", "tugas10_json") \
    .option("checkpointLocation", "tugas10_json_checkpoint") \
    .outputMode("append") \
    .start()

time.sleep(8)
query.stop()

In [18]:
!ls tugas10_json

df4 = spark.read.json("tugas10_json")
df4.show()

part-00000-11b4925d-098e-4a4a-9dce-a47fd283b443-c000.json
part-00001-dcc98b0d-28d7-4283-a65f-93c264623cff-c000.json
part-00002-46bde78a-2556-483b-ab01-d03ba558a1cf-c000.json
part-00003-de1b2287-e4a0-4c24-82ce-da14b0746602-c000.json
_spark_metadata
+------------+-----------+----------+----------+---+--------------------+
|harga_satuan|nama_cabang| nama_menu|pesanan_id|qty|         waktu_pesan|
+------------+-----------+----------+----------+---+--------------------+
|       15000|   Magelang| Kopi Susu|      P001|  2|2026-11-05T08:06:...|
|       18000| Yogyakarta| Americano|      P002|  1|2026-11-05T08:06:...|
|       20000|   Magelang|Cappuccino|      P003|  3|2026-11-05T08:06:...|
|       20000|   Semarang|Cappuccino|      P007|  1|2026-11-05T08:06:...|
|       18000|   Magelang| Americano|      P008|  2|2026-11-05T08:06:...|
|       22000| Yogyakarta|     Latte|      P009|  1|2026-11-05T08:06:...|
|       22000|   Magelang|     Latte|      P004|  1|2026-11-05T08:06:...|
|       1500

In [24]:
query1.stop()
query2.stop()
query3.stop()

spark.stop()